In [170]:
from types import SimpleNamespace

from ipywidgets import interact
import matplotlib
from matplotlib import pyplot as plt
from mpl_toolkits import mplot3d
import numpy as np

import kwant

import scipy.linalg as la
from math import atan2, pi, sqrt, exp

import scipy
import tinyarray
from IPython.display import display, Math
import warnings

warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

In [171]:
def plot_bands_BZ(syst, symmetry_points, step=0.1, params = None, fig_size = (14, 10), mode=1, Nbands=1):
    """Plot 2D band structure of a wrapped around system.

    This function is primarily useful for systems that have translational
    symmetry vectors that are non-orthogonal (e.g. graphene). This function
    will properly plot the band structure in an orthonormal basis in k-space,
    as opposed to in the basis of reciprocal lattice vectors (which would
    produce a "skewed" Brillouin zone).

    If your system has orthogonal lattice vectors, you are probably better
    off using `kwant.plotter.spectrum`.

    Parameters
    ----------
    syst : `kwant.system.FiniteSystem`
        A 2D system that was finalized from a Builder produced by
        `kwant.wraparound.wraparound`.  Note that this *must* be a finite
        system; so `kwant.wraparound.wraparound` should have been called with
        ``keep=None``.
    symmetry_points : array of arrays of symmetry points
        Symmtry point for which the bandstructure is to be calculated.
        The required format is an array of arrays of ["Label", Position in 2D k-space]
    step : step in units of k-vector that the vector path is divided with 
    params : dict, optional
        Dictionary of parameter names and their values, not including the
        momentum parameters.
    fig_size : tuple, optional
        Figure size `(width, height)` in inches.  If not set, the default
        ``matplotlib`` value is used.

    Returns
    -------
    fig : matplotlib figure

    Notes
    -----
    This function produces bandstructure plot: energy versus the momentum at given symmetry points
    A path given by the set os symmetry points is created and then divided into segments with a given step value.
    Bases on wraparound.plot_2d_bands to convert the basis of reciprocal lattice vectors
    in which the wrapped system is defined to a basis orthogonal in k-space

    """

    if not hasattr(syst, '_wrapped_symmetry'):
        raise TypeError("Expecting a system that was produced by "
                        "'kwant.wraparound.wraparound'.")
    
    if not isinstance(syst, kwant.system.FiniteSystem):
        msg = ("All symmetry directions must be wrapped around: specify "
               "'keep=None' when calling 'kwant.wraparound.wraparound'.")
        raise TypeError(msg)

    params = params or {}
    lat_ndim, space_ndim = syst._wrapped_symmetry.periods.shape

    if lat_ndim != 2:
        raise ValueError("Expected a system with a 2D translational symmetry.")
    if space_ndim != lat_ndim:
        raise ValueError("Lattice dimension must equal realspace dimension.")

    # columns of B are lattice vectors
    B = np.array(syst._wrapped_symmetry.periods).T
    # columns of A are reciprocal lattice vectors
    A = B.dot(np.linalg.inv(B.T.dot(B)))

    def momentum_to_lattice(k):
        k, residuals = scipy.linalg.lstsq(A, k)[:2]
        if np.any(abs(residuals) > 1e-7):
            raise RuntimeError("Requested momentum doesn't correspond"
                               " to any lattice momentum.")
        return k

    def ham(k_x, k_y=None, **params):
        # transform into the basis of reciprocal lattice vectors
        k = momentum_to_lattice([k_x] if k_y is None else [k_x, k_y])
        p = dict(zip(syst._momentum_names, k), **params)
        return syst.hamiltonian_submatrix(params=p, sparse=False)
    
    
    def calculate_energy(kx, ky):
        ret = np.sort(np.linalg.eigvalsh(ham(k_x=kx, k_y=ky, **params))).real
        
        return ret
    

    
        
    k_values = []
    energy_values = []
    labels = []
    label_points = []
    k_number = 0
    
    for i_point, k_point in enumerate(symmetry_points):
        #Save position and text for label
        labels.append(k_point[0])
        if (i_point == 0):
            label_points.append(k_number)
        elif (i_point > 0):
            #Calculate length of the vector between the symmetry points
            start_point = symmetry_points[i_point-1][1]
            end_point = k_point[1]
            
            path_vector = np.subtract(end_point, start_point)
            number_of_points_per_vector = int(np.linalg.norm(path_vector)/step)
            
            #Divide the vector into coordinate points
            x_translation_points = np.linspace(start_point[0], end_point[0], number_of_points_per_vector)
            y_translation_points = np.linspace(start_point[1], end_point[1], number_of_points_per_vector)
            
            #Iterate over the points and calculate the energies
            #If last vector include also the end point (otherwise ommited not do double points)
            if (i_point == len(symmetry_points) - 1):
                ommit_last_point = 0
            else:
                ommit_last_point = -1
            for point_number in range(number_of_points_per_vector+ommit_last_point):
                energy = calculate_energy(x_translation_points[point_number], y_translation_points[point_number])
                energy_values.append(energy)  
                k_values.append(k_number)
                k_number += 1
                
            label_points.append(k_number)

    label_points[-1] -= 1

    energy_values = np.array(energy_values, dtype = float)
    
    
    Nk = len(k_values)
    ktab = np.zeros(Nk)
    
    for i in range(Nk):
        ktab[i] = step*k_values[i]
        
    if(mode == 1 or mode == 2):
        fig, ax = pyplot.subplots(figsize = fig_size)
        pyplot.xticks(fontsize = 24)
        pyplot.yticks(fontsize = 24)
    
        if(mode == 1):    # we mark on the kx-axis the high symmetry points
            for i in range(Nbands):
                pyplot.plot(k_values, energy_values[:,i])  
            
            pyplot.xticks(label_points, labels)
    
        if(mode == 2):    # we mark on the kx-axis the k "distance"
            pyplot.plot(ktab, energy_values[:,0])
            pyplot.plot(ktab, energy_values[:,1])
    
        pyplot.grid()
        
        pyplot.ylabel("E [t]", fontsize = 24)
        #plt.ylim(-2,2)  

        return fig
    
    if(mode == 3):
        class disp:
            def __init__(self,ktab,energy_values):
                self.ktab = ktab
                self.energy_values  = energy_values
                
        disp1 = disp(ktab, energy_values)
        return disp1

In [172]:
#Here we define functions that will be used to build MoTe2/WSe2 systems

#The matrix elements are ordered in the following scheme:
#e_up h_up e_down h_down
def onsite_A_sup(site, Bz, delta, delta_sup, delta_sup_hopp, t_ab, faza, faza_AB, t_aa, t_bb, mu, sign):
    return np.array([[-delta/2 + Bz - mu, 0, 0, delta_sup],
                     [0,  +delta/2 - Bz + mu, sign*delta_sup, 0],
                     [0, sign*delta_sup, -delta/2 - Bz - mu, 0],
                     [delta_sup, 0, 0, +delta/2 + Bz + mu]])

def onsite_B_sup(site, Bz, delta, delta_sup, delta_sup_hopp, t_ab, faza, faza_AB, t_aa, t_bb, mu, sign):
    return np.array([[delta/2 + Bz - mu, 0, 0 , delta_sup],
                     [0, -delta/2 - Bz + mu, sign*delta_sup, 0],
                     [0, sign*delta_sup, delta/2 - Bz - mu, 0],
                     [delta_sup, 0, 0, -delta/2 + Bz + mu]])


def onsite_A(site, Bz, delta, delta_sup, delta_sup_hopp, t_ab, faza, faza_AB, t_aa, t_bb, mu, sign):
    return np.array([[-delta/2 + Bz - mu, 0, 0, 0],
                     [0, +delta/2 - Bz + mu, 0, 0],
                     [0, 0, -delta/2 - Bz - mu, 0],
                     [0, 0, 0, +delta/2 + Bz + mu]])

def onsite_B(site, Bz, delta, delta_sup, delta_sup_hopp, t_ab, faza, faza_AB, t_aa, t_bb, mu, sign):
    return np.array([[delta/2 + Bz - mu, 0, 0 , 0],
                     [0, -delta/2 - Bz + mu, 0, 0],
                     [0, 0, delta/2 - Bz - mu, 0],
                     [0, 0, 0, -delta/2 + Bz + mu]])



def hopping_AA(site1, site2, Bz, delta, delta_sup, delta_sup_hopp, t_ab, faza, faza_AB, t_aa, t_bb, mu, sign):
    t_AA_up = t_aa * np.exp(1j * 1 * faza)
    t_AA_down = t_aa * np.exp(1j * (-1) * faza)
    return np.array([[t_AA_up,0,0,0],
                     [0,-np.conjugate(t_AA_up),0,0],
                     [0,0,t_AA_down,0],
                     [0,0,0,-np.conjugate(t_AA_down)]])

def hopping_BB(site1, site2, Bz, delta, delta_sup, delta_sup_hopp, t_ab, faza, faza_AB, t_aa, t_bb, mu, sign ):
    t_BB_up = t_bb * np.exp(1j * 1 * faza)
    t_BB_down = t_bb * np.exp(1j * (-1) * faza)
    return np.array([[t_BB_up,0,0,0],
                     [0,-np.conjugate(t_BB_up),0,0],
                     [0,0,t_BB_down,0],
                     [0,0,0,-np.conjugate(t_BB_down)]])



def hopping_AA_SC(site1, site2, Bz, delta, delta_sup, delta_sup_hopp, t_ab, faza, faza_AB, t_aa, t_bb, mu, sign):
    t_AA_up = t_aa 
    t_AA_down = t_aa 
    return np.array([[t_AA_up,0,0,0],
                     [0,-t_AA_up,0,0],
                     [0,0,t_AA_down,0],
                     [0,0,0,-t_AA_down]])

def hopping_BB_SC(site1, site2, Bz, delta, delta_sup, delta_sup_hopp, t_ab, faza, faza_AB, t_aa, t_bb, mu, sign):
    t_BB_up = t_bb 
    t_BB_down = t_bb 
    return np.array([[t_BB_up,0,0,0],
                     [0,-t_BB_up,0,0],
                     [0,0,t_BB_down,0],
                     [0,0,0,-t_BB_down]])



#Table of directions of hoppings v_ij
v_ab = np.array([0, 0, 0])*(-1)
# v_ab = np.array([0, 1, 2])*(-1)
# v_ba = np.array([0, 1, 2])
v_ba = np.array([0, 0, 0])


def hopping_AB(site1, site2, Bz, delta, delta_sup, delta_sup_hopp, t_ab, t_ab_cons, faza, faza_AB, t_aa, t_bb, mu, sign):
    t_AB_up = t_ab * np.exp(1j * 1 * faza_AB)
    t_AB_down = t_ab * np.exp(1j * 1 * faza_AB)
    t_BA_up = t_ab * np.exp(1j * 1 * faza_AB)
    t_BA_down = t_ab * np.exp(1j * 1 * faza_AB)
    return np.array([[t_ab_cons, 0, t_AB_up, delta_sup_hopp],
                     [0, -np.conjugate(t_ab_cons), sign*delta_sup_hopp, -np.conjugate(t_AB_down)],
                     [t_AB_down, sign*delta_sup_hopp, t_ab_cons, 0],
                     [delta_sup_hopp, -np.conjugate(t_AB_up), 0, -np.conjugate(t_ab_cons)]])

def hopping_BA(site1, site2, Bz, delta, delta_sup, delta_sup_hopp, t_ab, t_ab_cons, faza, faza_AB, t_aa, t_bb, mu, sign):
    t_AB_up = t_ab * np.exp(1j * 1 * faza_AB)
    t_AB_down = t_ab * np.exp(1j * 1 * faza_AB)
    t_BA_up = t_ab * np.exp(1j * 1 * faza_AB)
    t_BA_down = t_ab * np.exp(1j * 1 * faza_AB)
    return np.array([[t_ab_cons, 0, t_AB_up, delta_sup_hopp],
                     [0, -np.conjugate(t_ab_cons), sign*delta_sup_hopp, -np.conjugate(t_AB_down)],
                     [t_AB_down, sign*delta_sup_hopp, t_ab_cons,  0],
                     [delta_sup_hopp, -np.conjugate(t_AB_up), 0, -np.conjugate(t_ab_cons)]])



def hopping_AB_SC(site1, site2, Bz, delta, delta_sup, delta_sup_hopp, t_ab, faza, faza_AB, t_aa, t_bb, mu, sign):
    t_AB_up = t_ab 
    t_AB_down = t_ab
    t_BA_up = t_ab 
    t_BA_down = t_ab
    return np.array([[t_AB_up,0,0,0],
                     [0,-t_AB_up,0,0],
                     [0,0,t_AB_down,0],
                     [0,0,0,-t_AB_down]])


def hopping_BA_SC(site1, site2, Bz, delta, delta_sup, delta_sup_hopp, t_ab, faza, faza_AB, t_aa, t_bb, mu, sign):
    t_AB_up = t_ab 
    t_AB_down = t_ab
    t_BA_up = t_ab 
    t_BA_down = t_ab 
    return np.array([[t_AB_up,0,0,0],
                     [0,-t_AB_up,0,0],
                     [0,0,t_AB_down,0],
                     [0,0,0,-t_AB_down]])


In [173]:
#Function that creates zigzag MoTe2/WSe2 system
def build_graphene_system_zigzag_2(Lmin=-4, Lmax=4, Wmin=-10, Wmax=10):
    
    graphene_lat = kwant.lattice.general([[1, 0], [1/2, np.sqrt(3)/2]],  # lattice vectors
                                 [[0, 0], [0, 1/np.sqrt(3)]], norbs = 4)  # Coordinates of the sites
    lat_A, lat_B = graphene_lat.sublattices


    sys_gr1 = kwant.Builder(kwant.TranslationalSymmetry(*graphene_lat.prim_vecs))
    sys_gr1[graphene_lat.shape((lambda pos: True), (0, 0))] = 0 #?


    def ribbon(pos):
        x, y = pos
        return Lmin <= x <= Lmax and Wmin <= y <= Wmax
    
    
    sys_gr1[lat_A(0,0)] = onsite_A_sup
    sys_gr1[lat_B(0,0)] = onsite_B_sup
    

    sys_gr1[kwant.builder.HoppingKind((0, 0), lat_A, lat_B)] = hopping_AB
    sys_gr1[kwant.builder.HoppingKind((0, -1), lat_A, lat_B)] = hopping_AB
    sys_gr1[kwant.builder.HoppingKind((1, -1), lat_A, lat_B)] = hopping_AB


    sys_gr1[kwant.builder.HoppingKind((-1, 0), lat_A, lat_A)] = hopping_AA
    sys_gr1[kwant.builder.HoppingKind((0, 1), lat_A, lat_A)] = hopping_AA
    sys_gr1[kwant.builder.HoppingKind((1, -1), lat_A, lat_A)] = hopping_AA

    sys_gr1[kwant.builder.HoppingKind((-1, 0), lat_B, lat_B)] = hopping_BB
    sys_gr1[kwant.builder.HoppingKind((0, 1), lat_B, lat_B)] = hopping_BB
    sys_gr1[kwant.builder.HoppingKind((1, -1), lat_B, lat_B)] = hopping_BB

   
    
    sysf_gr1 = kwant.wraparound.wraparound(sys_gr1).finalized()

    
    return sysf_gr1

In [176]:
# In this cell, we plot the bandstructure of zigzag MoTe2/WSe2 system
#sysf_zigzag_2 = None
sysf_zigzag_2 = build_graphene_system_zigzag_2()

def plot_wf2(Bz=0, delta= 13, t_ab=3, t_ab_cons=0, t_aa = 2, delta_sup = 0, delta_sup_hopp = 0, sign = -1):
#almost all parameters are defined here. They are passed to function "build_graphene_system" and then to onsite and hopping functions
    t_bb=t_aa
    faza = 2*np.pi/3
    faza_AB = 0
    mu = 2.5
    global sysf_zigzag_2
    sysf_zigzag_2 = build_graphene_system_zigzag_2()
    
    params = dict(Bz=Bz, delta=delta, delta_sup = delta_sup, delta_sup_hopp = delta_sup_hopp, t_ab = t_ab, t_ab_cons = t_ab_cons, faza = faza, faza_AB = faza_AB, t_aa = t_aa, t_bb = t_bb, mu = mu, sign = sign)
#     kwant.wraparound.plot_2d_bands(sysf_zigzag_2, k_x = 100, k_y = 100, fig_size = (12,12), params= params);
    

    symmetry_points1 = [["Γ", [0,0]],
                       ["K'", [2*np.pi/(3), 2*np.pi*np.sqrt(3)/(3)]],
                       ["M", [2*np.pi/(2), 2*np.pi*np.sqrt(3)/(6)]],
                       ["K", [2*np.pi*2/(3), 0]],
                       ["Γ", [0,0]]]   

    params1 = plot_bands_BZ(sysf_zigzag_2, params = params, symmetry_points = symmetry_points1, step=0.01, mode=3, Nbands=8)

    plt.title('in kx')
    plt.plot((params1.ktab), params1.energy_values[:,0],'b')  
    plt.plot((params1.ktab), params1.energy_values[:,1],'orange')  
    plt.plot((params1.ktab), params1.energy_values[:,2],'g')
    plt.plot((params1.ktab), params1.energy_values[:,3],'r')  
    plt.plot((params1.ktab), params1.energy_values[:,4],'violet')  
    plt.plot((params1.ktab), params1.energy_values[:,5],'yellow')
    plt.plot((params1.ktab), params1.energy_values[:,6],'black')
    plt.plot((params1.ktab), params1.energy_values[:,7],'brown')

    plt.ylim(-40,40)
    plt.xlim(0,np.pi*4)


    plt.grid(1)
    plt.figure()


interact(plot_wf2, Bz=(-10.0, 10.0), delta=(-60.0, 60.0), t_ab=(-10.0, 10.0), t_ab_cons=(-10.0, 10.0), t_aa=(-10.0, 10.0), delta_sup=(-10.0, 10.0), delta_sup_hopp = (-10.0, 10.0), sign = (-1, 1));

interactive(children=(FloatSlider(value=0.0, description='Bz', max=10.0, min=-10.0), FloatSlider(value=13.0, d…